In [1]:
# import pathlib

import pandas as pd

import plotly
import plotly.express as px
import plotly.graph_objects as go

# from IPython.display import display, Markdown, Latex

import trapallada.analysis

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

So that *plotly* pictures can be exported to HTML

In [ ]:
plotly.offline.init_notebook_mode()

# Input Files

Relative path and *Base* file name for the results (trained model). It should be a `str`.

In [4]:
input_files_stem = '../results/epochs/20000/4_sources/epochs'

In [ ]:
ds, positions_df, model, results_df, parameters, analysis_df = trapallada.analysis.load_results_dir(input_files_stem, analysis_also=True)

In [ ]:
analysis_df.head()

For the sake of convenience

In [7]:
# pos_columns = ['RA', 'DEC', 'CRA', 'CDEC']
tsne_columns = [f'tsne_{i}' for i in range(1, 3)]

The number of objects for every *type*

In [ ]:
analysis_df['main_type'].value_counts()

The clustering (either *in projected space* or *in z*) to be analyzed

In [9]:
cluster = 'cluster in projected space'
# cluster = 'cluster in z'

A few `main_type`s are selected

In [ ]:
on_plot_hover_columns = ['index', 'source', cluster, 'main_type']
on_plot_hover_columns

In [11]:
for c in ['cluster in projected space', 'cluster in z']:
    analysis_df[cluster] = pd.Categorical(analysis_df[cluster])
    # display(analysis_df[c].value_counts())

# All the signals

Empty `main_type`s are set to `UNKNOWN`

In [12]:
analysis_df['main_type'].fillna('UNKNOWN', inplace=True)

In order to have fixed axes

In [ ]:
# analysis_df[tsne_columns].min(), analysis_df[tsne_columns].max()
xrange, yrange = list(zip(analysis_df[tsne_columns].min(), analysis_df[tsne_columns].max()))
xrange, yrange

One can **double-click** a `main_type` in the *Legend* to only show the corresponding curves

In [ ]:
fig = px.scatter(
    analysis_df.reset_index(),
    *tsne_columns,
    color='main_type',
    hover_data=on_plot_hover_columns, opacity=0.5,
    width=1_200, height=800)
fig.show()

A convenience function

In [15]:
def plot_rectangle(top_left_corner: tuple[float], bottom_right_corner: tuple[float], n_samples: int = 5):

    # `True` if the tSNE coordinates of the corresponding point are inside the rectangle
    selected = (analysis_df['tsne_1'] > top_left_corner[0]) &\
        (analysis_df['tsne_1'] < bottom_right_corner[0]) &\
        (analysis_df['tsne_2'] < top_left_corner[1]) &\
        (analysis_df['tsne_2'] > bottom_right_corner[1])

    # the original `DataFrame` is subset
    selected_df = analysis_df[selected]

    # a copy of the above figure
    fig_with_rectangle = go.Figure(fig)

    # the rectangle is overimposed on the picture
    fig_with_rectangle.add_shape(
        type='rect',
        x0=top_left_corner[0], x1=bottom_right_corner[0],
        y0=top_left_corner[1], y1=bottom_right_corner[1],
        line=dict(color='RoyalBlue'),
        fillcolor='LightSkyBlue',
        opacity=0.4
    )

    fig_with_rectangle.show()

    for i, series in selected_df.sample(n_samples).iterrows():
        
        display(trapallada.analysis.predict_and_plot(model, ds, i))

# South cluster

Top-left and bottom-right of the rectangle

In [16]:
top_left_corner = (-30., -53.)
bottom_right_corner = (6.5, -78.)

In [ ]:
plot_rectangle(top_left_corner, bottom_right_corner, n_samples=20)

# North-east cluster

Top-left and bottom-right of the rectangle

In [18]:
top_left_corner = (33.8, 53.5)
bottom_right_corner = (56.8, 32.)

In [ ]:
plot_rectangle(top_left_corner, bottom_right_corner, n_samples=20)

# North cluster

Top-left and bottom-right of the rectangle

In [20]:
top_left_corner = (17.8, 65.5)
bottom_right_corner = (28., 55.8)

In [ ]:
plot_rectangle(top_left_corner, bottom_right_corner, n_samples=20)